# Jupyter notebook UI

In [1]:
import vscodenb
import crossrepo
import pandas as pd

## Make results accessible by `crossref`

To register files or directories as data available to `crossrepo`, add a `crossrepo.yml` to a `results` folder at the root of your repository. Keys are file names, paths or globs and values say what the file holds. You can list parquet folders as if they were files, since they load as such.

### GitHub repositories

To register files or directories as data available to `crossrepo`, add a `crossrepo.yml` to the repository `results`. Keys are file names, paths or globs and values say what the file holds. You can list parquet folders as if they were files, since they load as such.

```yml
files:
  hits.csv: Genome-wide association hits, p < 5e-8
  qc/summary.csv: Per-sample genotyping quality
  by_chrom: Per-chromosome effect sizes, one file per chromosome
```

A key beginning with `/` is a path from the repository root rather than from the
`crossrepo.yml`, which publishes a file that is kept elsewhere in the repository.
Any tracked file can be named this way; `..` cannot be used to climb out.

```yml
files:
  hits.csv: In the results directory, as usual
  /data/reference/samples.csv: Somewhere else in the repository
  /data/raw/*.tsv: A pattern, matched against the path from the root
```

The easiest way to add a file or folder is using the `crossrepo share` command:

```bash
crossrepo share results/your_file.csv 'Short description of the file'
```

### Local copies of repositories

`crossrepo` can also be pointed to local copies of repos with available assets. If `crossrepo` finds a local copy on the same file system, it will make a hardlink to the asset rather than downloading it from GitHub. This is instant and takes up no extra space on disk.

### Large files not tracked by git

Accessing working copies is useful if they contain temporary result files too big for github commit. In that case, you add a symlink (E.g. `dummy.csv -> ../steps/out/dummy.csv`) to your results folder and then add the symlink to `crossrepo.yml`:

```yml
files:
  dummy.csv: Dummy csv file sym linked from steps
```

However, since `git` only tracks the relative path of the symlink and not the file it links to, you need to generate a version identifier for the linked file. You do that by running `crossrepo stamp` in the repository. This adds a version hash to `crossrepo.yml`. Once you have committed the updated `crossrepo.yml`, the large file is available through `crossrepo`.

```yml
files:
  dummy.csv:
    description: Dummy csv file sym linked from steps
    sha256: "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
    size: 0
```

`crossrepo share` makes this a bit easier. If the file is a symlink, `crossrepo` will automatically run the `stamp` command too.


```bash
ln -s steps/large.txt results
git add results/large.txt && git commit -m 'Added symlink' 
crossrepo share results/symlink_to untracked_file.csv 'Short description of untracked the file'
```


## Config

`crossrepo` configuration is read from a `crossrepo.toml` file in the current directory and otherwise from `~/.config/crossrepo/config.toml`. You create a `~/.config/crossrepo/config.toml` by running:

```bash
crossrepo config global init
```

and a local `crossrepo.yml` by replacing `global` with `local`.

In [2]:
crossrepo.active_config()

Config(roots=[],
       owners=['munch-group'],
       repos=[],
       asset_dirs=['results', 'data'],
       include=['*.csv',
                '*.tsv',
                '*.txt',
                '*.parquet',
                '*.pq',
                '*.h5',
                '*.hdf',
                '*.hdf5',
                '*.store',
                '*.json',
                '*.jsonl',
                '*.xlsx',
                '*.bed',
                '*.gff',
                '*.vcf',
                '*.vcf.gz',
                '*.pkl',
                '*.pickle',
                '*.npy',
                '*.npz',
                '*.feather',
                '*.zarr'],
       exclude=['*.png',
                '*.pdf',
                '*.svg',
                '*.html',
                '*.md',
                '.gitkeep',
                '*.log'],
       min_bytes=0,
       max_bytes=0)

You can also generate a config on the fly that is only specific to one notebook:

In [3]:
cfg = crossrepo.config.Config(
    repos=['munch-group/relate1Kgenomes', 'munch-group/atlas-variant-ages'], 
    include=['*.parquet', '*.csv', '*.tsv'],
    exclude=['.store', '*.fa', '*.vcf', '*.bim', '*.fam', '*.ped', '*.bam'],
)
crossrepo.use_config(cfg)

Config(roots=[],
       owners=[],
       repos=['munch-group/relate1Kgenomes', 'munch-group/atlas-variant-ages'],
       asset_dirs=['results'],
       include=['*.parquet', '*.csv', '*.tsv'],
       exclude=['.store', '*.fa', '*.vcf', '*.bim', '*.fam', '*.ped', '*.bam'],
       min_bytes=0,
       max_bytes=0)

You an alway see the active configs using:

In [4]:
crossrepo.active_config() 

Config(roots=[],
       owners=[],
       repos=['munch-group/relate1Kgenomes', 'munch-group/atlas-variant-ages'],
       asset_dirs=['results'],
       include=['*.parquet', '*.csv', '*.tsv'],
       exclude=['.store', '*.fa', '*.vcf', '*.bim', '*.fam', '*.ped', '*.bam'],
       min_bytes=0,
       max_bytes=0)

To go activate the default configs specified in your config file, just pass `None` to `use_config`:

In [5]:
crossrepo.use_config(None) 

Config(roots=[],
       owners=['munch-group'],
       repos=[],
       asset_dirs=['results', 'data'],
       include=['*.csv',
                '*.tsv',
                '*.txt',
                '*.parquet',
                '*.pq',
                '*.h5',
                '*.hdf',
                '*.hdf5',
                '*.store',
                '*.json',
                '*.jsonl',
                '*.xlsx',
                '*.bed',
                '*.gff',
                '*.vcf',
                '*.vcf.gz',
                '*.pkl',
                '*.pickle',
                '*.npy',
                '*.npz',
                '*.feather',
                '*.zarr'],
       exclude=['*.png',
                '*.pdf',
                '*.svg',
                '*.html',
                '*.md',
                '.gitkeep',
                '*.log'],
       min_bytes=0,
       max_bytes=0)

A cache refresh is run the first time you use the it after you change the configs. You can always force a cache refresh using the `refresh` method:

In [6]:
crossrepo.refresh()

reading GitHub:   0%|          | 0/157 [00:00<?, ?repo/s]

### GitHub remote repositories

Search all github repos under these accounts listed in `owners` and/or in each repo listed in `repos`:


In [7]:
cfg = crossrepo.config.Config(
    owners=["munch-group", ], 
    repos=["songlab-cal/gpn", 'MyersGroup/relate'], 
)
crossrepo.use_config(cfg)

Config(roots=[],
       owners=['munch-group'],
       repos=['songlab-cal/gpn', 'MyersGroup/relate'],
       asset_dirs=['results'],
       include=[],
       exclude=[],
       min_bytes=0,
       max_bytes=0)

### Local working copies

You can use `roots` to list folders where you have checked out working copies of repositories. When files exist on the same filesystem, `crossrepo.get()` returns a hardlink rather than a copy to not take up extra disk space.

In [8]:
cfg = crossrepo.config.Config(
    roots = ["~/", ]
)
crossrepo.use_config(cfg)

Config(roots=['~/'],
       owners=[],
       repos=[],
       asset_dirs=['results'],
       include=[],
       exclude=[],
       min_bytes=0,
       max_bytes=0)

Working copies of repositories takes precedence over remotes found on github.  

### Remote working copies

You can also access working copies of repositories on remote servers, like GenomeDK, if you have set up password-less access using ssh keys. To make two-factor authentication with the genome.dk cluster more smooth, you can add this to your `~/.ssh/config` (replacing "kmt" with your own username on the cluster):

```txt
Host *
  ServerAliveInterval 60

Host gdk
    HostName        login.genome.au.dk
    User            kmt
    ControlMaster   auto
    ControlPath     ~/.ssh/cm-%r@%h:%p
    ControlPersist  4h
```

and then use the `gdk` alias when specifying a root in configs:

## Browsing the catalog

In [9]:
cfg = crossrepo.config.Config(
    owners=["munch-group"], 
    repos=["songlab-cal/gpn", 'MyersGroup/relate'],     
    roots = ["gdk:xy-drive/people/kmt", "gdk:primatediversity/people/kmt", "~/"]
)
crossrepo.use_config(cfg)

Config(roots=['gdk:xy-drive/people/kmt',
              'gdk:primatediversity/people/kmt',
              '~/'],
       owners=['munch-group'],
       repos=['songlab-cal/gpn', 'MyersGroup/relate'],
       asset_dirs=['results'],
       include=[],
       exclude=[],
       min_bytes=0,
       max_bytes=0)

`crossrepo` will refresh the catalog of assets scan the sources to update the catalog of assets if this more than an hour old, or if the configs have changed. You can always force a refresh with:

In [10]:
crossrepo.refresh()

scanning clones:   0%|          | 0/42 [00:00<?, ?repo/s]

reading GitHub:   0%|          | 0/159 [00:00<?, ?repo/s]

(same as `crossrepo refresh` command).

List the catalog accessible using the active configs:

In [11]:
crossrepo.list()

,repo,path,get,description
0,munch-group/atlas-variant-ages,results/atlas_variant_ages.parquet,github,Variant ages using Abers method
1,munch-group/crossrepo,results/dummy.csv,local,Dome dummy csv file sym linked from steps
2,munch-group/crossrepo,results/dummy.txt,github,Dummy text file
3,munch-group/crossrepo,results/large.txt,local,Tester symlinked file
4,munch-group/hic-xy-sperm,results/all_genes.h5,github,all genes
5,munch-group/hic-xy-sperm,results/segments_100000.csv,github,100kb segments
6,munch-group/hic-xy-sperm,results/segments_50000.csv,github,50kb segments
7,munch-group/hic-xy-sperm,results/segments_500000.csv,github,500kb segments
8,munch-group/relate1Kgenomes,results/relate_snps_p_vals.parquet,github,Genome SNPs (hg38) with a log p-value below -2w
9,munch-group/vep-data,results/vep.parquet,gdk,Parquet formatted VEP info from ensembl


(same as `crossrepo list` command).

Use `repos` to list repos searched for catalog assets:

In [12]:
crossrepo.repos()

,repo,files,bytes,latest
0,munch-group/atlas-variant-ages,1,1890901982,2026-09-06
1,munch-group/crossrepo,3,24,2026-09-12
2,munch-group/hic-xy-sperm,4,8260043,2026-09-06
3,munch-group/relate1Kgenomes,1,82477487,2026-09-06
4,munch-group/vep-data,1,31840511289,2026-09-11


(same as `crossrepo repos` command).

Use `info` to list information about an asset:

In [13]:
crossrepo.info("munch-group/relate1Kgenomes", "results/relate_snps_p_vals.parquet")

munch-group/relate1Kgenomes:results/relate_snps_p_vals.parquet

description  Genome SNPs (hg38) with a log p-value below -2w
get          github
path         results/relate_snps_p_vals.parquet
manifest     results/crossrepo.yml
root         gdk:xy-drive/people/kmt/relate1Kgenomes

version      f2a7042fde239562518974d7b29a61fe94910cf5
date         2026-09-06
commit       update
size         78.7M
parts        3
content      03215a0b27dfcfa4796ce1f43c8d9efbed1c6f53
url          https://github.com/munch-group/relate1Kgenomes/tree/f2a7042fde239562518974d7b29a61fe94910cf5/results/relate_snps_p_vals.parquet


As long as the repo names and file names are unique, you can skip the owner and folder information:

In [14]:
crossrepo.info("relate1Kgenomes", "relate_snps_p_vals.parquet")

munch-group/relate1Kgenomes:results/relate_snps_p_vals.parquet

description  Genome SNPs (hg38) with a log p-value below -2w
get          github
path         results/relate_snps_p_vals.parquet
manifest     results/crossrepo.yml
root         gdk:xy-drive/people/kmt/relate1Kgenomes

version      f2a7042fde239562518974d7b29a61fe94910cf5
date         2026-09-06
commit       update
size         78.7M
parts        3
content      03215a0b27dfcfa4796ce1f43c8d9efbed1c6f53
url          https://github.com/munch-group/relate1Kgenomes/tree/f2a7042fde239562518974d7b29a61fe94910cf5/results/relate_snps_p_vals.parquet


(same as `crossrepo info relate1Kgenomes relate_snps_p_vals.parquet` command).

Use `get` to the path to a downloaded/hardlinked asset.

In [15]:
tmp_path = crossrepo.get("relate1Kgenomes", "relate_snps_p_vals.parquet")

Add version="f2a7042fde239562518974d7b29a61fe94910cf5" to pin this version.


(same as `crossrepo get relate1Kgenomes relate_snps_p_vals.parquet` command).

Now silent, because this is still the current version

In [16]:
tmp_path = crossrepo.get("relate1Kgenomes", "relate_snps_p_vals.parquet",
                         version="f2a7042fde239562518974d7b29a61fe94910cf5")

In [17]:
df = pd.read_parquet(tmp_path)
df.head()

,chrom,pop,pos,p_half_freq,p_two_alleles
0,chr1,ACB,817341,-0.473037,-2.88930
1,chr1,ACB,892733,-1.231820,-2.30420
2,chr1,ACB,893360,-1.231820,-2.30420
3,chr1,ACB,897538,-1.319140,-4.10702
4,chr1,ACB,901516,-1.319140,-3.19570


In [18]:
filters = [
    ('pos', '>=', 892733), 
    ('pos', '<', 901516),
    ('pop', '==', "ACB"),
    ('chrom', '==', "chrX"),
]
pd.read_parquet(tmp_path, filters=filters)

,chrom,pop,pos,p_half_freq,p_two_alleles
0,chrX,ACB,893285,-1.273310,-4.51818
1,chrX,ACB,895075,-1.713310,-4.48943
2,chrX,ACB,895885,-1.274320,-2.17077
3,chrX,ACB,896099,-1.274320,-2.17077
4,chrX,ACB,896561,-1.713310,-4.48943
5,chrX,ACB,897491,-1.355870,-5.26528
6,chrX,ACB,897556,-0.750727,-3.91539
7,chrX,ACB,897658,-0.574665,-2.00685
8,chrX,ACB,898415,-0.301048,-4.08054
9,chrX,ACB,899070,-0.574665,-2.00685


## Version history

The catalog stamps every file with the repository's current commit.
`crossrepo.versions()` shows the commits in which the file itself changed, which
is the useful set to pin.

In [19]:
crossrepo.versions("relate1Kgenomes", "relate_snps_p_vals.parquet")

,version,date,bytes,parts,tags,subject
0,f4f1fc621cb9dfac3610f7f47d90caf640b43571,2026-09-01,82477487,3,,Added parquet fixed data
